# Part 3: Multi-GPU Training with DDP — a Scaling Study

**Learning objectives.** By the end of this session you will be able to:

- Launch a PyTorch DistributedDataParallel (DDP) job with `torchrun` under SLURM
- Explain what DDP replicates, shards, and synchronizes
- Measure strong-scaling efficiency and identify what limits it

**How this part works.** Training happens *outside* the notebook, in `train_ddp.py`, launched by `submit_ddp.sh`. You submit the job at 1, 2, and 4 GPUs; each run appends a line to `scaling_results.csv`; this notebook analyzes the results. Notebooks are for exploration and  analysis — batch scripts are for compute. 

## What DDP actually does

Each GPU gets a full copy of the model and a *shard* of each batch (`DistributedSampler`). Every backward pass ends with an all-reduce that averages gradients across GPUs, so all copies stay identical. Compute scales with GPU count; the all-reduce is the serial-ish part that Amdahl's law punishes. Read `train_ddp.py` before running it — it is ~100 lines and every DDP-specific line is commented.

Submit the scaling study:

    sbatch --gres=gpu:1 submit_ddp.sh
    sbatch --gres=gpu:2 submit_ddp.sh
    sbatch --gres=gpu:4 submit_ddp.sh

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    results = np.genfromtxt("scaling_results.csv", delimiter=",",
                            names=True, encoding="utf-8")
    results = np.atleast_1d(results)
    n_gpus = results["n_gpus"]
    throughput = results["samples_per_sec"]
    print("Loaded", len(results), "runs")
except OSError:
    # Placeholder so the notebook renders before the jobs finish.
    print("scaling_results.csv not found - using illustrative numbers.")
    n_gpus = np.array([1, 2, 4])
    throughput = np.array([1.0e6, 1.85e6, 3.2e6])

In [ ]:
order = np.argsort(n_gpus)
n_gpus, throughput = n_gpus[order], throughput[order]
speedup = throughput / throughput[0]
efficiency = speedup / n_gpus

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.plot(n_gpus, speedup, "o-", lw=2, color="darkorange", label="Measured")
ax.plot(n_gpus, n_gpus, "k--", lw=1, label="Ideal (linear)")
ax.set_xlabel("Number of GPUs")
ax.set_ylabel("Speedup vs 1 GPU")
ax.set_title("Strong scaling")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar([str(int(g)) for g in n_gpus], 100 * efficiency, color="steelblue")
ax.axhline(100, color="k", ls="--", lw=1)
ax.set_xlabel("Number of GPUs")
ax.set_ylabel("Parallel efficiency [%]")
ax.set_title("Efficiency = speedup / n_gpus")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Interpreting your numbers

Efficiency below 100% has three usual suspects: the gradient all-reduce
(communication), data loading (CPU-side bottleneck), and small per-GPU
batches that leave the GPU underfed. Our model is tiny, so expect
*mediocre* scaling — the all-reduce is cheap but so is the compute it
overlaps with. That is a feature of this exercise, you learn more from
diagnosing imperfect scaling.

### Exercise 4 — feed the GPUs

Double the per-GPU batch size in `submit_ddp.slurm` (`--batch-size 2048`)
and rerun the study. Predict before you run: will efficiency go up or
down, and why? Then check. As a follow-up, try `--hidden 1024` to make
the model compute-heavier and watch scaling improve.

### Exercise 5 — mixed precision

Rerun with `--amp`. On modern GPUs (V100 onward) matrix multiplies in
float16/bfloat16 use tensor cores. How much throughput does it buy for
this model, and why is the gain smaller than for a large transformer?